In [0]:
volume_path = "/Volumes/capstone_project_07/capstone_07_schema/capstone_07_ext_vol"

display(dbutils.fs.ls(volume_path))

In [0]:
display(
    dbutils.fs.ls(
        f"{volume_path}/cdc_data/document_metadata_changes"
    )
)

In [0]:
folders = {
    "CDC": f"{volume_path}/cdc_data/document_metadata_changes",
    "Access Audit": f"{volume_path}/event_data/document_access_audit.csv",
    "Search Events": f"{volume_path}/event_data/search_query_events.csv",
    "Category Lookup": f"{volume_path}/lookup_data/azure_sql_seed",
    "User Master": f"{volume_path}/master_data/azure_sql_seed"
}

for dataset, path in folders.items():

    print(f"\n========== {dataset} ==========")

    files = dbutils.fs.ls(path)

    for file in files:
        print(file.name, file.size)

In [0]:
from pyspark.sql.functions import col, element_at, split, count, lit

# Directories to count (each read as CSV with a header row)
directories = {
    "cdc_data/document_metadata_changes": f"{volume_path}/cdc_data/document_metadata_changes",
    "event_data": f"{volume_path}/event_data",
    "lookup_data/azure_sql_seed": f"{volume_path}/lookup_data/azure_sql_seed",
    "master_data/azure_sql_seed": f"{volume_path}/master_data/azure_sql_seed",
}

per_file = None
for dir_name, path in directories.items():
    df = (
        spark.read.option("header", True).csv(path)
        .select(element_at(split(col("_metadata.file_path"), "/"), -1).alias("file_name"))
        .groupBy("file_name").agg(count(lit(1)).alias("record_count"))
        .withColumn("directory", lit(dir_name))
    )
    per_file = df if per_file is None else per_file.unionByName(df)

# Per-directory totals
display(
    per_file.groupBy("directory")
    .agg({"record_count": "sum", "file_name": "count"})
    .withColumnRenamed("sum(record_count)", "total_records")
    .withColumnRenamed("count(file_name)", "file_count")
    .orderBy("directory")
)

# Per-file breakdown
display(per_file.select("directory", "file_name", "record_count").orderBy("directory", "file_name"))